# Model Blend — NNLS over base learners + saturation residual

**Inputs (from previous notebooks)**

| File | Source | Contains |
|---|---|---|
| `oof/lgbm_huber.parquet` etc. | `model_train_lgbm` | OOF predictions on train, three diverse LGBM objectives |
| `oof/moe_blended.parquet`, `moe_expert.parquet`, `moe_global.parquet` | `model_train_moe` | OOF predictions from the RoadType MoE |
| `oof/saturation_prob.parquet` | `model_train_saturation` | P(demand≥0.99) on saturating-geohash rows |
| `predictions/*_test.parquet` | same notebooks | test predictions |
| `oof/sat_fire_mask_train.npy`, `predictions/sat_fire_mask_test.npy` | saturation notebook | bool mask of where the residual correction applies |

**Plan**

1. Merge the 6 regression-OOF columns into a single wide dataframe aligned by `Index`.
2. Fit **NNLS** (`scipy.optimize.nnls`) with X = the 6 OOFs and y = `demand`. Non-negativity is exactly what we want — it prevents nonsense weights, naturally zeros out redundant models, and gives an interpretable mixture.
3. Compute OOF blend prediction on every train row, clip to `[0, 1]`, and report:
   - `val_fold R²` of each base individually
   - `val_fold R²` of the blend without saturation residual
   - `val_fold R²` of the blend WITH saturation residual
   Pick whichever variant is higher.
4. Apply the same weights and (optionally) saturation correction to the test predictions.
5. Save `predictions/final_test_predictions.parquet` for inference notebook.

> The saturation correction is `pred + P_sat * (1 - pred)` applied ONLY on rows in the fire mask (Highway-saturating geohashes with `NumberofLanes_modal >= 2`).


## 1. Setup + load

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.optimize import nnls
from sklearn.metrics import r2_score

BASE = '.'
PROCESSED = os.path.join(BASE, 'processed')
OOF_DIR   = os.path.join(BASE, 'oof')
PRED_DIR  = os.path.join(BASE, 'predictions')

train = pd.read_parquet(os.path.join(PROCESSED, 'train_features.parquet'))
test  = pd.read_parquet(os.path.join(PROCESSED, 'test_features.parquet'))
val_split = pd.read_parquet(os.path.join(PROCESSED, 'val_split_v2_indices.parquet'))
train = train.merge(val_split.rename(columns={'row_id':'Index'}), on='Index', how='left')
val_mask_train = (train['fold'] == 'val_fold').values

BASE_MODELS = ['lgbm_huber','lgbm_mae','lgbm_quantile','moe_blended','moe_expert','moe_global']
print('Base models:', BASE_MODELS)

Base models: ['lgbm_huber', 'lgbm_mae', 'lgbm_quantile', 'moe_blended', 'moe_expert', 'moe_global']


## 2. Build wide OOF + test prediction tables

In [2]:
def load_oof(name):
    p = pd.read_parquet(os.path.join(OOF_DIR, f'{name}.parquet'))
    p = p.rename(columns={f'{name}_oof_pred': name})
    return p[['Index', name]]

def load_test_pred(name):
    p = pd.read_parquet(os.path.join(PRED_DIR, f'{name}_test.parquet'))
    p = p.rename(columns={f'{name}_test_pred': name})
    return p[['Index', name]]

oof_df = train[['Index','demand']].copy()
for m in BASE_MODELS:
    oof_df = oof_df.merge(load_oof(m), on='Index', how='left')

test_pred_df = test[['Index']].copy()
for m in BASE_MODELS:
    test_pred_df = test_pred_df.merge(load_test_pred(m), on='Index', how='left')

n_nan_oof = oof_df[BASE_MODELS].isna().sum().sum()
n_nan_test = test_pred_df[BASE_MODELS].isna().sum().sum()
print(f'OOF NaN total: {n_nan_oof}, test pred NaN total: {n_nan_test}')
assert n_nan_oof == 0 and n_nan_test == 0, 'NaN in base predictions — fix upstream'
print('Wide tables built ✓')
oof_df.head(3)

OOF NaN total: 0, test pred NaN total: 0
Wide tables built ✓


,Index,demand,lgbm_huber,lgbm_mae,lgbm_quantile,moe_blended,moe_expert,moe_global
0,0,0.048804,0.019525,0.017949,0.059880,0.021849,0.021294,0.022460
1,1,0.118507,0.020694,0.023132,0.087421,0.023894,0.024769,0.021296
2,2,0.027132,0.016824,0.028717,0.065498,0.018743,0.021064,0.013561


## 3. Per-base R² (sanity)

In [3]:
print('=== Per-base OOF R² (all train) and val_fold OOF R² ===')
for m in BASE_MODELS:
    all_r2 = r2_score(oof_df['demand'], oof_df[m])
    val_r2 = r2_score(oof_df.loc[val_mask_train, 'demand'], oof_df.loc[val_mask_train, m])
    print(f'  {m:18s}  all={all_r2:.5f}  val_fold={val_r2:.5f}')

=== Per-base OOF R² (all train) and val_fold OOF R² ===
  lgbm_huber          all=0.95761  val_fold=0.96653
  lgbm_mae            all=0.95331  val_fold=0.96479


  lgbm_quantile       all=0.91533  val_fold=0.92920
  moe_blended         all=0.95678  val_fold=0.96576
  moe_expert          all=0.95504  val_fold=0.96434
  moe_global          all=0.95787  val_fold=0.96669


## 4. NNLS blend

In [4]:
X = oof_df[BASE_MODELS].values
y = oof_df['demand'].values
w, residual = nnls(X, y)
weights = {m: float(wi) for m, wi in zip(BASE_MODELS, w)}
sum_w = sum(weights.values())
print('NNLS weights:')
for m, wi in weights.items():
    print(f'  {m:18s} {wi:.4f}')
print(f'  sum = {sum_w:.4f}, residual norm = {residual:.4f}')

NNLS weights:
  lgbm_huber         0.2304
  lgbm_mae           0.0000
  lgbm_quantile      0.0572
  moe_blended        0.0000
  moe_expert         0.2531
  moe_global         0.4699
  sum = 1.0105, residual norm = 8.0205


In [5]:
blend_oof = np.clip(X @ w, 0.0, 1.0)
blend_oof_r2_all = r2_score(y, blend_oof)
blend_oof_r2_val = r2_score(y[val_mask_train], blend_oof[val_mask_train])
print(f'Blend OOF R²: all={blend_oof_r2_all:.5f}, val_fold={blend_oof_r2_val:.5f}')

Blend OOF R²: all=0.95884, val_fold=0.96720


## 5. Saturation residual correction (try both with and without)

In [6]:
sat_train_p = pd.read_parquet(os.path.join(OOF_DIR, 'saturation_prob.parquet'))
sat_test_p  = pd.read_parquet(os.path.join(PRED_DIR, 'saturation_prob_test.parquet'))
sat_train_p = sat_train_p.set_index('Index')['sat_prob_oof']
sat_test_p  = sat_test_p.set_index('Index')['sat_prob_test']

fire_train = np.load(os.path.join(OOF_DIR, 'sat_fire_mask_train.npy'))
fire_test  = np.load(os.path.join(PRED_DIR, 'sat_fire_mask_test.npy'))
print(f'Fire mask: train {fire_train.sum()}, test {fire_test.sum()}')

# Align sat_prob to OOF row order
oof_df['sat_prob_oof'] = oof_df['Index'].map(sat_train_p)
sat_oof = oof_df['sat_prob_oof'].fillna(0.0).values
test_pred_df['sat_prob_test'] = test_pred_df['Index'].map(sat_test_p)
sat_test = test_pred_df['sat_prob_test'].fillna(0.0).values

# Apply correction on fire mask only
blend_oof_sat = blend_oof.copy()
blend_oof_sat[fire_train] = np.clip(
    blend_oof[fire_train] + sat_oof[fire_train] * (1.0 - blend_oof[fire_train]),
    0.0, 1.0
)
sat_oof_r2_all = r2_score(y, blend_oof_sat)
sat_oof_r2_val = r2_score(y[val_mask_train], blend_oof_sat[val_mask_train])
print(f'Blend+Sat OOF R²: all={sat_oof_r2_all:.5f}, val_fold={sat_oof_r2_val:.5f}')

use_sat = sat_oof_r2_val > blend_oof_r2_val
print(f'\nUse saturation correction? {use_sat}'
      f' (delta val R² = {sat_oof_r2_val - blend_oof_r2_val:+.5f})')

Fire mask: train 1890, test 846
Blend+Sat OOF R²: all=0.95502, val_fold=0.96688

Use saturation correction? False (delta val R² = -0.00032)


## 6. Apply to test

In [7]:
X_test = test_pred_df[BASE_MODELS].values
blend_test = np.clip(X_test @ w, 0.0, 1.0)
if use_sat:
    blend_test[fire_test] = np.clip(
        blend_test[fire_test] + sat_test[fire_test] * (1.0 - blend_test[fire_test]),
        0.0, 1.0
    )
print(f'Test prediction stats: min={blend_test.min():.5f}, max={blend_test.max():.5f}, '
      f'mean={blend_test.mean():.5f}, std={blend_test.std():.5f}')

Test prediction stats: min=0.00354, max=0.99965, mean=0.08029, std=0.11652


## 7. Save final test predictions + blend metadata

In [8]:
final = pd.DataFrame({'Index': test_pred_df['Index'].values, 'final_demand': blend_test})
final.to_parquet(os.path.join(PRED_DIR, 'final_test_predictions.parquet'), index=False)

blend_info = dict(
    weights=weights,
    weight_sum=float(sum_w),
    val_fold_r2={
        'lgbm_huber':     float(r2_score(oof_df.loc[val_mask_train,'demand'], oof_df.loc[val_mask_train,'lgbm_huber'])),
        'lgbm_mae':       float(r2_score(oof_df.loc[val_mask_train,'demand'], oof_df.loc[val_mask_train,'lgbm_mae'])),
        'lgbm_quantile':  float(r2_score(oof_df.loc[val_mask_train,'demand'], oof_df.loc[val_mask_train,'lgbm_quantile'])),
        'moe_blended':    float(r2_score(oof_df.loc[val_mask_train,'demand'], oof_df.loc[val_mask_train,'moe_blended'])),
        'moe_expert':     float(r2_score(oof_df.loc[val_mask_train,'demand'], oof_df.loc[val_mask_train,'moe_expert'])),
        'moe_global':     float(r2_score(oof_df.loc[val_mask_train,'demand'], oof_df.loc[val_mask_train,'moe_global'])),
        'blend_no_sat':   float(blend_oof_r2_val),
        'blend_with_sat': float(sat_oof_r2_val),
    },
    all_train_r2={
        'blend_no_sat':   float(blend_oof_r2_all),
        'blend_with_sat': float(sat_oof_r2_all),
    },
    use_saturation=bool(use_sat),
)
with open(os.path.join(BASE, 'blend_weights.json'), 'w') as f:
    json.dump(blend_info, f, indent=2)
print('Saved predictions/final_test_predictions.parquet and blend_weights.json')

print(f'\n*** Estimated competition score: {sat_oof_r2_val*100 if use_sat else blend_oof_r2_val*100:.2f} ***')
print('(Based on val_fold OOF R², which is the best honest estimate we have.)')

Saved predictions/final_test_predictions.parquet and blend_weights.json

*** Estimated competition score: 96.72 ***
(Based on val_fold OOF R², which is the best honest estimate we have.)
